# inf-masking — faded example 1: Fill the -inf causal mask fill

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `inf-masking`. Running the beacon reports progress on the `Numpy: Inf-fill masking trick` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Inf-fill masking trick` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`inf-masking`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "inf-masking"
DD_SUBTOPIC = "Numpy: Inf-fill masking trick"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

Causal attention fills the future positions (strictly upper triangle) with `-inf` before softmax so their post-softmax weight is exactly zero.

## Faded exercise 1

Implement `causal_softmax(scores)` for `(T, T)` scores. The causal mask is given; fill the masked positions with `-inf` and softmax over the last axis. Complete the blanked masked-fill-then-softmax expression.

**Fill in:** the masked_fill(mask, -inf) followed by softmax over the key axis

In [ ]:
import torch as t

t.manual_seed(3)
scores = t.randn(5, 5)

def causal_softmax(scores):
    T = scores.shape[-1]
    mask = t.triu(t.ones(T, T, dtype=t.bool), diagonal=1)
    out = scores.masked_fill(mask, float('-inf')).softmax(dim=-1)
    return out

print(causal_softmax(scores).sum(-1))


def _test():
    w = causal_softmax(scores)
    assert w.shape == (5, 5)
    # future positions must be EXACTLY zero (independent of softmax numerics)
    assert bool((w.triu(diagonal=1) == 0).all())
    # each row is a valid distribution over allowed keys
    assert t.allclose(w.sum(-1), t.ones(5), atol=1e-6)
    # row 0 attends only to key 0
    assert abs(float(w[0, 0]) - 1.0) < 1e-6


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

t.manual_seed(3)
scores = t.randn(5, 5)

def causal_softmax(scores):
    T = scores.shape[-1]
    mask = t.triu(t.ones(T, T, dtype=t.bool), diagonal=1)
    out = scores.masked_fill(mask, float('-inf')).softmax(dim=-1)
    return out

print(causal_softmax(scores).sum(-1))
```
</details>